In [47]:
using Pkg
Pkg.activate("../UDEs")
Pkg.status()

In [48]:
using Revise, Optimization, ModelingToolkit,DifferentialEquations,Plots, Lux, PEtab, Random
seed = 0

0

### Lotka-Volterra
We simulate the Lotka-Volterra equation

In [49]:
## Lotka-Volterra equations
@parameters α β γ δ
@independent_variables  t
@variables x(t) y(t) z(t)
Dt = Differential(t)
eqs = [
    Dt(x) ~ α * x - β * x * y,
    Dt(y) ~ δ * x * y - γ * y,
]
measured_quantities = [z ~ x + y]  # Example of a measured quantity
@named sys = ODESystem(eqs, t, [x, y], [α, β, γ, δ]; observed = measured_quantities)
sys = complete(sys)
params =  Dict([α => 0.1, 
                β => 0.02, 
                δ => 0.01,
                γ => 0.3])

u0 = Dict([x => 40.0, y => 9.0])
tspan = (0.0, 200.0)
dt = 0.1

sys = complete(sys)
odefun = ODEFunction(sys, unknowns(sys), parameters(sys))
prob = ODEProblem(odefun, [40.0, 9.0], tspan, [0.1, 0.02, 0.01, 0.3])
sol = solve(prob, Tsit5(), saveat=dt)
data = hcat(sol.u...)'
plot(sol, vars=(x, y), xlabel="prey", ylabel="predator",
     title="Lotka-Volterra Model", label="Solution",
     legend=:topright, linewidth=2, markersize=4)


### Random noise
We add random noise to the Lotka-Volterra equations to simulate a more realistic scenario.


In [50]:
using Random
rng = Random.default_rng()
Random.seed!(seed)
noise = 0.2 * randn(size(data))
data = data .+ noise
time = sol.t
plot(time, data, label=["Prey" "Predator"], xlabel="Time", ylabel="Population", title="Lotka-Volterra Model with Noise")

In [51]:

### ENSEMBLE DATA ###

initial_conditions  = [
    [40.0, 9.00],
    [20.0, 8.0],
    [10, 7.0], 
]
function prob_func(prob, i, repeat)
    remake(prob, u0 = initial_conditions[i])
end

ensemble_prob = EnsembleProblem(prob, prob_func = prob_func)
sim = solve(ensemble_prob, Tsit5(), EnsembleDistributed(), trajectories = 3, saveat = time)
# ensemble_data = hcat([sim.u[1].u; sim.u[2].u; sim.u[3].u]...)
#ensemble_data = ensemble_data .+ randn(size(ensemble_data)) * 0.02 # Add noise to the ensemble data'
plot(sim)
#save dat as named tuple



MethodError: MethodError: no method matching EnsembleProblem(::ODEProblem{Vector{Float64}, Tuple{Float64, Float64}, true, Vector{Float64}, ODEFunction{true, SciMLBase.AutoSpecialize, ModelingToolkit.GeneratedFunctionWrapper{(2, 3, true), RuntimeGeneratedFunctions.RuntimeGeneratedFunction{(:__mtk_arg_1, :___mtkparameters___, :t), ModelingToolkit.var"#_RGF_ModTag", ModelingToolkit.var"#_RGF_ModTag", (0x715036cc, 0x067c60b1, 0xbb64e06c, 0x84aee536, 0xf39695ce), Nothing}, RuntimeGeneratedFunctions.RuntimeGeneratedFunction{(:ˍ₋out, :__mtk_arg_1, :___mtkparameters___, :t), ModelingToolkit.var"#_RGF_ModTag", ModelingToolkit.var"#_RGF_ModTag", (0x6180adc8, 0xfda1678a, 0xcf53bed6, 0xa7c6bf38, 0x5077a38d), Nothing}}, LinearAlgebra.UniformScaling{Bool}, Nothing, Nothing, Nothing, Nothing, Nothing, Nothing, Nothing, Nothing, Nothing, Nothing, Nothing, ModelingToolkit.ObservedFunctionCache{ODESystem}, Nothing, ODESystem, Nothing, Nothing}, Base.Pairs{Symbol, Union{}, Tuple{}, @NamedTuple{}}, SciMLB

In [52]:
using Catalyst
using KolmogorovArnold
using ComponentArrays

In [53]:
using Lux
# Gaussian RBF as activation
rbf(x) = exp.(-(x.^2))

nn1 = Lux.Chain(
    Lux.Dense(2,5,rbf; init_weight =  kaiming_normal),
    Lux.Dense(5,5, rbf, init_weight = kaiming_normal),
    Lux.Dense(5,5, rbf, init_weight = kaiming_normal),
    Lux.Dense(5,2, rbf, init_weight = kaiming_normal),
)
p_nn, st_nn = Lux.setup(rng, nn1)

((layer_1 = (weight = Float32[1.3630719 0.1947098; 0.41142717 0.68750584; … ; -1.1411679 2.4023454; 1.1907109 -0.13519815], bias = Float32[0.005757597, -0.16040273, -0.07710917, 0.5509642, 0.41567793]), layer_2 = (weight = Float32[0.503292 -0.28269142 … -0.4981909 1.3524925; -0.53077364 -0.09820266 … -0.21933961 0.19988945; … ; -0.5694064 -0.12438205 … -0.07181738 -0.7761605; -0.021173079 -0.012916612 … 0.255255 0.82405], bias = Float32[-0.06620707, 0.27438205, 0.2915852, -0.34449407, -0.14109527]), layer_3 = (weight = Float32[-0.24117601 -0.031014888 … -0.507782 -0.80591065; -0.09679303 0.27807584 … -0.19423454 0.37743282; … ; 0.72122437 0.062242236 … -0.15925862 0.17285216; 1.076888 0.16130222 … 0.38019636 -0.665267], bias = Float32[0.37084398, 0.2430911, -0.07927683, -0.11521985, -0.10247812]), layer_4 = (weight = Float32[0.045020234 0.4664588 … -0.15542017 0.000605549; -0.08404695 -0.3500733 … 0.69674057 -0.015944613], bias = Float32[0.43282372, 0.0028493672])), (layer_1 = NamedTup

In [54]:
basis_func = KolmogorovArnold.rbf # rbf, rswaf, iqf (radial basis funcs, reflection switch activation funcs, inverse quadratic funcs)
normalizer = softsign # sigmoid(_fast), tanh(_fast), softsign
kan1 = Chain(
    KDense( 2, 40, 10; use_base_act = true, basis_func, normalizer),
    KDense(40, 40, 10; use_base_act = true, basis_func, normalizer),
    KDense(40,  2, 10; use_base_act = true, basis_func, normalizer),
) # 18_490 parameters plus 30 states.
p_kan, st_kan = Lux.setup(rng, kan1)
kan1([40.0, 9.0], p_kan, st_kan)

([-2.081205109247073, -1.5683368729794105], (layer_1 = (grid = Float32[-1.0, -0.7777778, -0.5555556, -0.33333334, -0.11111111, 0.11111111, 0.33333334, 0.5555556, 0.7777778, 1.0],), layer_2 = (grid = Float32[-1.0, -0.7777778, -0.5555556, -0.33333334, -0.11111111, 0.11111111, 0.33333334, 0.5555556, 0.7777778, 1.0],), layer_3 = (grid = Float32[-1.0, -0.7777778, -0.5555556, -0.33333334, -0.11111111, 0.11111111, 0.33333334, 0.5555556, 0.7777778, 1.0],)))

In [55]:
kan1

Chain(
    layer_1 = KDense{true, typeof(softsign), Tuple{Float32, Float32}, Float32, typeof(KolmogorovArnold.rbf), typeof(swish), typeof(glorot_uniform), typeof(glorot_uniform)}(2, 40, 10, NNlib.softsign, (-1.0f0, 1.0f0), 0.22222222f0, KolmogorovArnold.rbf, NNlib.swish, WeightInitializers.glorot_uniform, WeightInitializers.glorot_uniform),  # 880 parameters, plus 10
    layer_2 = KDense{true, typeof(softsign), Tuple{Float32, Float32}, Float32, typeof(KolmogorovArnold.rbf), typeof(swish), typeof(glorot_uniform), typeof(glorot_uniform)}(40, 40, 10, NNlib.softsign, (-1.0f0, 1.0f0), 0.22222222f0, KolmogorovArnold.rbf, NNlib.swish, WeightInitializers.glorot_uniform, WeightInitializers.glorot_uniform),  # 17_600 parameters, plus 10
    layer_3 = KDense{true, typeof(softsign), Tuple{Float32, Float32}, Float32, typeof(KolmogorovArnold.rbf), typeof(swish), typeof(glorot_uniform), typeof(glorot_uniform)}(40, 2, 10, NNlib.softsign, (-1.0f0, 1.0f0), 0.22222222f0, KolmogorovArnold.rbf, NNlib.swish

### Hidden Dynamics with UDEs

In [56]:
known_eqs = [
    Dt(x) ~ α * x
    Dt(y) ~ -γ * y
]
unknown_eqs = [
    Dt(x) ~ -β * x * y,
    Dt(y) ~ δ * x * y,
]
deviance = 0.1 # deviance for the unknown equations
params_guess_known = Dict([α => 0.1, # + deviance * randn()
                        γ => 0.3, # + deviance * randn()])
                        ])

params_guess_unknown = Dict([β => 0.02,# + deviance * randn()
                        δ => 0.01 # + deviance * randn()])
                        ])


@named sys_known = ODESystem(known_eqs, t, [x, y], [α, γ], defaults = params_guess_known, observed = measured_quantities)
@named sys_unknown = ODESystem(unknown_eqs, t, [x, y], [β, δ], defaults = params_guess_unknown)
sys_known = complete(sys_known)
sys_unknown = complete(sys_unknown)

Model sys_unknown:
Equations (2):
  2 standard: see equations(sys_unknown)
Unknowns (2): see unknowns(sys_unknown)
  x(t)
  y(t)
Parameters (2): see parameters(sys_unknown)
  β [defaults to 0.02]
  δ [defaults to 0.01]

In [57]:
import ModelingToolkit:has_observed,observables
import Optimization:solve

mutable struct NullModel <: ModelingToolkit.AbstractTimeDependentSystem
    """ A null model that does nothing. Used as a placeholder for surrogate models."""
    name::String

    """ Create a NullModel with a given name."""
    NullModel(name::String = "NullModel") = new(name)

    """ Get the parameters of the NullModel. Returns an empty NamedTuple."""
end

""" HybridModel is a system that combines an ODE system with a surrogate model (SINDy or neural network)."""
mutable struct HybridModel
    """ The known underlying ODE system."""
    sys::ODESystem
    """ The surrogate model, which can be a normal ODESystem, SINDy model, or a Lux neural network."""
    surrogate::Union{ModelingToolkit.AbstractTimeDependentSystem, Lux.Chain}
    """ Discrete events that trigger during the simulation."""
    events::Vector
    """ Observables that are computed during the simulation. (Defaults to unknowns of sys)"""
    observables::Vector
    """ Random number generator for reproducibility."""
    rng ::Random.AbstractRNG
    """ODEFunction for the HybridModel system."""
    ode_fun::Function

    
    """ Construct a HybridModel system with a SINDy/ODE surrogate model. """
    HybridModel(sys::ODESystem, surrogate::T;
                events::Vector = [], 
                observables::Vector = has_observed(sys) || has_observed(surrogate) ?  union([observables(sys);observables(surrogate)]) : unknowns(sys), #choose observables from sys or surrogate if they have them. else use state variables
                rng::Random.AbstractRNG = Random.default_rng(1234),
                ode_fun::Function = ODEFunction(sys, surrogate; rng = rng)
                ) where 
                {T <: ModelingToolkit.AbstractTimeDependentSystem} = 
        new(sys, surrogate, events, observables, rng, ode_fun)


    """   Construct a HybridModel system with a Lux neural network surrogate model."""
    HybridModel(sys::ODESystem, surrogate::T,; 
               events::Vector = [], 
               observables::Vector = has_observed(sys) || has_observed(surrogate) ?  union([observables(sys);observables(surrogate)]) : unknowns(sys), #choose observables from sys or surrogate if they have them. else use state variables
               rng::Random.AbstractRNG = Random.default_rng(1234),
               ode_fun::Function = ODEFunction(sys, surrogate; rng = rng)
               ) where 
               {T <: Lux.Chain} =    
        new(sys, surrogate, events, observables, rng, ode_fun)
end

# Convert a dictionary to a NamedTuple (for use with ComponentArrays with ModelingToolkit)
NamedTuple(dict::Dict) = (; (Symbol(string(k)) => v for (k, v) in dict)...) 

#### HIDDEN ODE METHODS ####

function init_params(model::HybridModel)
    @unpack sys, surrogate, rng = model
    
    # Initialize parameters for the ODE system
    ode_ps = init_params(sys)
    # Initialize parameters for the surrogate model
    surrogate_ps = init_params(surrogate; rng)
    
    # Combine the parameters into a NamedTuple
    combined_ps = (; ode = ode_ps, surrogate = surrogate_ps)
    return ComponentVector{Float64}(combined_ps)
end

function ODEFunction(model::HybridModel)
        @unpack sys, surrogate, rng = model
        return ODEFunction(sys, surrogate; rng = rng)
end

#create the ODE function for the HybridModel system
function ODEFunction(sys::ODESystem, surrogate::T; rng = Random.default_rng(1234)) where {T <: Union{ModelingToolkit.AbstractTimeDependentSystem, Lux.Chain}}
    # Get the derivative function for the ODE system
    ode_fun! = ODEFunction(sys)
    # Get the surrogate derivative function
    surrogate_fun! = derivative_function!(surrogate; rng = rng)

    du1 = zeros(length(unknowns(sys))) # Initialize du1 for the ODE system
    du2 = copy(du1) # Initialize du2 for the surrogate model
    function update_du!(du, u, p, t)
        # Compute the ODE derivatives
        ode_fun!(du1, u, p.ode, t) 
        # Compute the surrogate derivatives
        surrogate_fun!(du2, u, p.surrogate, t) 
        # Combine the derivatives   
        du.= du1 .+ du2 
        return du  
    end
    odefun! = remake(ode_fun!, f = update_du!) # Create a new ODEFunction with the combined derivatives
    return odefun!
end

function ODEProblem(model::HybridModel, u0::Vector, tspan, p = init_params(model))
    @unpack ode_fun = model
    prob = ODEProblem(ode_fun, u0, tspan, p)
end

function EnsembleProblem(model::HybridModel, u0s::Vector{Vector{T}}, tspan, p = init_params(model)) where {T <: Any}
    prob = ODEProblem(model, u0s[1], tspan, p)
    function prob_func(prob, i, repeat)
        remake(prob, u0 = u0s[i])
    end
    return EnsembleProblem(prob, prob_func = prob_func)
end

function solve(model::HybridModel, u0s, time, p = init_params(model); alg = Tsit5, kwargs...)
    tspan = (time[1], time[end])
    prob = EnsembleProblem(model, u0s, tspan, p)
    # if prob isa EnsembleProblem
    return solve(prob, alg(), EnsembleDistributed(), trajectories = length(u0s), saveat = time, kwargs...)
    # elseif prob isa ODEProblem
    #     return solve(prob, alg(), saveat = time, kwargs...)
    # end
end

function observed_values(model::HybridModel, sol)
    @unpack observables = model
    return observed_values(observables, sol)
end

function observed_values(observed::Vector, sol)
    # Get the observed values from the solution
    return ComponentArray(; (Symbol(string("iv$i")) => hcat([sim[var] for var in observed]...) for (i, sim) in enumerate(sol))...)
end


### ODE SYSTEM METHODS ###
function init_params(sys::ODESystem; randfun = rand, rng = Random.default_rng(1234))
    if ModelingToolkit.has_defaults(sys)
        defaults = ModelingToolkit.get_defaults(sys)
        params = parameters(sys)
        # Initialize parameters with defaults (NamedTuple))
        return (; (Symbol(string(p)) => defaults[p] for p in parameters(sys) if p in keys(defaults))...)
    else
        # If no defaults, return an empty random values
        return (; (Symbol(string(p)) => randfun() for p in parameters(sys))...)
    end
end


function derivative_function!(sys::ODESystem; rng = Random.default_rng(1234))
    return ODEFunction(sys)
end

### NEURAL NETWORK METHODS ###
function init_params(nn::Lux.Chain; rng = Random.default_rng(1234))
    # Get the parameters of the Lux model
    return Lux.initialparameters(rng, nn)
end

function derivative_function!(nn::Lux.Chain; rng = Random.default_rng(1234))
    # Create a function that computes the derivatives of the Lux model
    st = Lux.initialstates(rng, nn)
    #NeuralODE. The output of the neural network is a vector of derivatives. Network only depends on state and network parameters.
    du = (du, u, p, t) -> first(nn(u, p, st))
    return deepcopy(du)
end

function has_observed(nn::Lux.Chain)
    # Check if the Lux model has observed quantities
    return false # Lux models do not have observed quantities by default
end

function observables(nn::Lux.Chain)
    # Lux models do not have observed quantities by default
    return []
end


### NULL MODEL METHODS ###
function init_params(model::NullModel; rng = Random.default_rng(1234))
    # Null model has no parameters (empty NamedTuple)
    return ()
end

function derivative_function!(model::NullModel; rng = Random.default_rng(1234))
    # Null model does nothing, so return a function that does nothing
    return (du, u, p, t) -> du .= 0.0
end

function has_observed(model::NullModel)
    # Null model has no observed quantities
    return false
end

function observables(model::NullModel)
    # Null model has no observed quantities
    return []
end



observables (generic function with 3 methods)

In [58]:
# hmodel = HybridModel(sys_known, sys_unknown; rng = rng)

# initp = init_params(hmodel)
# ### ENSEMBLE DATA ###

# initial_conditions  = [
#     [40.0, 9.00],
#     [20.0, 8.0],
#     [10, 7.0], 
# ]

# sim = solve(hmodel, initial_conditions, time, initp)
# plot(sim)


In [59]:
# solve(model::HiddenODE, u0, time, p = init_params(model); alg = Tsit5, kwargs...)
sol = solve(hmodel, initial_conditions, time, initp; alg = Tsit5)
plot(sol, vars=(x, y), xlabel="Prey", ylabel="Predator",
     title="Lotka-Volterra Model with Hidden ODE", label="Solution",
     legend=:topright, linewidth=2, markersize=4)

MethodError: MethodError: no method matching EnsembleProblem(::ODEProblem{Vector{Float64}, Tuple{Float64, Float64}, true, ComponentVector{Float64, Vector{Float64}, Tuple{Axis{(ode = ViewAxis(1:2, Axis(α = 1, γ = 2)), surrogate = ViewAxis(3:4, Axis(β = 1, δ = 2)))}}}, ODEFunction{true, SciMLBase.AutoSpecialize, var"#update_du!#78"{Vector{Float64}, Vector{Float64}, ODEFunction{true, SciMLBase.AutoSpecialize, ModelingToolkit.GeneratedFunctionWrapper{(2, 3, true), RuntimeGeneratedFunctions.RuntimeGeneratedFunction{(:__mtk_arg_1, :___mtkparameters___, :t), ModelingToolkit.var"#_RGF_ModTag", ModelingToolkit.var"#_RGF_ModTag", (0xe29672b4, 0x4b24ff6d, 0x715d2b3b, 0x86a38826, 0xd3010da3), Nothing}, RuntimeGeneratedFunctions.RuntimeGeneratedFunction{(:ˍ₋out, :__mtk_arg_1, :___mtkparameters___, :t), ModelingToolkit.var"#_RGF_ModTag", ModelingToolkit.var"#_RGF_ModTag", (0xc22018fd, 0xfd57c1d7, 0x9c91befa, 0xc014d706, 0x526fb6a9), Nothing}}, LinearAlgebra.UniformScaling{Bool}, Nothing, Nothing, No

In [60]:
event = PEtabEvent(x <= 1, 0, x)

PEtabEvent: Condition x(t) <= 1 and affect x(t) = 0

### PEtab - optimization 

In [61]:
#save model directory
model_dir = joinpath(@__DIR__, "models", "lotka_volterra", "regression","seed_$(seed)")
#boolean to indicate whether to load models or train models
load_models = true 
model_dir

"c:\\Users\\MGAJ\\OneDrive - Danmarks Tekniske Universitet\\DTU\\Kandidat\\5_Semester\\Speciale\\discovering_hidden_physics\\scripts\\models\\lotka_volterra\\regression\\seed_0"

In [62]:
#Convert data to dataframe
using DataFrames
n_data = size(data, 1)
sample_size = 200


sample_idcs = rand(1:n_data, sample_size)
prey_df = DataFrame(simulation_id = "cond1", obs_id = "prey_o", time = time[sample_idcs], measurement= data[sample_idcs, 1],)
predator_df = DataFrame(simulation_id = "cond1", obs_id = "predator_o", time = time[sample_idcs], measurement= data[sample_idcs, 2],)
measurements = vcat(prey_df, predator_df)

Row,simulation_id,obs_id,time,measurement
,String,String,Float64,Float64
1,cond1,prey_o,155.7,23.9928
2,cond1,prey_o,51.0,18.1544
3,cond1,prey_o,161.3,18.3731
4,cond1,prey_o,104.9,41.7586
5,cond1,prey_o,1.7,33.939
6,cond1,prey_o,164.2,18.293
7,cond1,prey_o,7.5,19.7947
8,cond1,prey_o,137.7,31.7785
9,cond1,prey_o,127.1,18.592


In [63]:
#show samples
scatter(time[sample_idcs], data[sample_idcs, :], label=["Prey" "Predator"], xlabel="Time", ylabel="Population", title="Lotka-Volterra Model with Sampled Data")

In [64]:
#Setup observables
@parameters σ
obs_x = PEtabObservable(x, σ)
obs_y = PEtabObservable(y, σ)
obs = Dict("prey_o" => obs_x, "predator_o" => obs_y)


#setup initial conditions
cond1 = Dict(:x => 40.0, :y => 9.0)
conds = Dict("cond1" => cond1)
# Setup parameters

#model parameters

estimate = true # Set to true if you want to estimate the parameters
p_α = PEtabParameter(α, lb = 1e-6, ub = 1e0, estimate = estimate, scale = :lin, value = 0.1)
p_β = PEtabParameter(β, lb = 1e-6 , ub = 1e0, estimate = estimate, scale = :lin, value = 0.02)
p_γ = PEtabParameter(γ, lb = 1e-6, ub = 1e0, estimate = estimate, scale = :lin, value = 0.3)
p_δ = PEtabParameter(δ, lb = 1e-6, ub = 1e0, estimate = estimate, scale = :lin, value = 0.01)
#noise parameter
p_σ = PEtabParameter(σ, lb = 1e-6, ub = 1e0, estimate = true, scale = :lin, value = 0.2)
pest = [
    p_α, p_β, p_γ, p_δ, p_σ
]
model = PEtabModel(sys,obs, measurements, pest; simulation_conditions  = conds)
petab_prob = PEtabODEProblem(model)

PEtabODEProblem: ODESystemModel with ODE-states 2 and 5 parameters to estimate
---------------- Problem options ---------------
Gradient method: ForwardDiff
Hessian method: ForwardDiff
ODE-solver nllh: Rodas5P
ODE-solver gradient: Rodas5P

In [ ]:
using Optim
n_runs = 20 # Number of runs for multistart optimization
if load_models
    # Load the models from the model directory
    res = PEtabMultistartResult(model_dir; which_run = 1)
else
    # Run the multistart optimization
    nprocs = 1 # Set the number of processes for parallel optimization
    res = calibrate_multistart(petab_prob, IPNewton(), n_runs, options = Optim.Options(show_trace = true, iterations = 1000, show_every = 10), nprocs = nprocs; seed = seed, save_trace = true, dirsave = model_dir)
end

PEtabMultistartResult
---------------- Summary ---------------
min(f)                = -6.41e+01
Parameters estimated  = 5
Number of multistarts = 20
Optimiser algorithm   = Optim_IPNewton
Results saved at c:\Users\MGAJ\OneDrive - Danmarks Tekniske Universitet\DTU\Kandidat\5_Semester\Speciale\discovering_hidden_physics\scripts\models\lotka_volterra\regression\seed_0


In [68]:
using Plots
using PEtab

In [69]:
function best_runs(res_ms, n)
    best_idxs = sortperm(getfield.(res_ms.runs, :fmin))
    return best_idxs[1: min(n, res_ms.nmultistarts)]
end

best_runs (generic function with 1 method)

In [70]:
idxs= best_runs(res, 10)
bestrun = res.runs[12]

PEtabOptimisationResult
---------------- Summary ---------------
min(f)                = -6.41e+01
Parameters estimated  = 5
Optimiser iterations  = 55
Runtime               = 1.2e+00s
Optimiser algorithm   = Optim_IPNewton


In [72]:
n_traces = 10 # Number of traces to plot #### DOESN*T WORK FOR SOME REASON
# plot(res; plot_type=:objective, best_idxs_n= 10)#,yscale =:log10, markersize = 4, xlim = (20, 100))#, idxs = best_runs(res, n_traces), yticks = 10.0 .^ (-1:9), best_idxs_n = 10) #

10

In [73]:
# plot(res; plot_type=:waterfall)
# plot(res; plot_type=:parallel_coordinates)

In [74]:
ds = get_obs_comparison_plots(res, petab_prob)["cond1"]["prey_o"]
ds

In [75]:
best_idxs = best_runs(res, 7)

7-element Vector{Int64}:
 12
 13
  9
 19
  4
 15
 11

In [76]:
ground_truth = [0.1, 0.3]
alphacoord = zeros(length(res.runs))
gammacoord = zeros(length(res.runs))
colors = [:blue, :green, :orange, :red, :purple, :cyan, :magenta, :yellow, :brown]
markers = [:v, :cross, :star, :triangle, :x, :diamond]
p1 = plot()
scatter!(p1, [ground_truth[1]], [ground_truth[2]], label = "Ground Truth", markersize = 20, color = :black, marker = :star, alpha = 0.5)
alpha = 0.8
for i in best_idxs
    color = colors[i % length(colors) + 1]
    marker = markers[i % length(markers) + 1]
    run = res.runs[i]
    p0 = run.x0
    alphacoord = p0.α
    gammacoord = p0.γ
    alpha_trace = hcat(run.xtrace ...)[1,:]
    gamma_trace = hcat(run.xtrace ...)[3,:]

    scatter!(p1, [alphacoord], [gammacoord], label = "initial $i", markersize = 4, color = color, marker = :circle, alpha = alpha)
    #plot the traces
    plot!(p1, alpha_trace, gamma_trace, label = "trace $i", linewidth = 1, markersize = 2, color = color, alpha = alpha)
    #mark the last point of the trace
    scatter!(p1, [alpha_trace[end]], [gamma_trace[end]], label = "stop $i", markersize = 10, color = color, marker = marker, alpha = alpha)
end

plot!(p1, xlabel = "α", ylabel = "γ", title = "Interior point Newton's Method", legend = :outertopright, xscale = :lin, yscale = :lin, markersize = 8)
#ground truth alpha and gamma
display(p1)

In [77]:
best_res = res.runs[best_idxs]

7-element Vector{PEtabOptimisationResult}:
 PEtabOptimisationResult
---------------- Summary ---------------
min(f)                = -6.41e+01
Parameters estimated  = 5
Optimiser iterations  = 55
Runtime               = 1.2e+00s
Optimiser algorithm   = Optim_IPNewton

 PEtabOptimisationResult
---------------- Summary ---------------
min(f)                = -6.41e+01
Parameters estimated  = 5
Optimiser iterations  = 80
Runtime               = 1.4e+00s
Optimiser algorithm   = Optim_IPNewton

 PEtabOptimisationResult
---------------- Summary ---------------
min(f)                = 5.84e+02
Parameters estimated  = 5
Optimiser iterations  = 63
Runtime               = 7.9e-01s
Optimiser algorithm   = Optim_IPNewton

 PEtabOptimisationResult
---------------- Summary ---------------
min(f)                = 1.23e+04
Parameters estimated  = 5
Optimiser iterations  = 43
Runtime               = 8.5e-01s
Optimiser algorithm   = Optim_IPNewton

 PEtabOptimisationResult
---------------- Summary -----

In [78]:
#parameter convergence plot

function param_trace(res, param_idx; ground_truth_value = nothing, run_idcs = collect(1:length(res.runs)),kwargs...)
    p1 = plot(; kwargs...)
    runs = res.runs[run_idcs]
    colors = [:blue, :green, :orange, :purple, :cyan, :magenta, :yellow, :brown]
    markers = [:v, :cross, :star, :triangle, :x]
    if ground_truth !== nothing
        hline!(p1, [ground_truth_value], label = "Ground Truth", color = :red, linestyle = :dash, linewidth = 6, alpha = 0.5)
    end
    for (i, run) in enumerate(runs)
        p0 = run.x0
        color = colors[i % length(colors) + 1]
        marker = markers[i % length(markers) + 1]
        param_trace = hcat(run.xtrace ...)[param_idx,:]
        scatter!(p1, param_trace, label = "Run $(run_idcs[i])", markersize = 4, color = color, marker = marker, alpha = 0.8)
        #plot the line connecting the points
        plot!(p1, param_trace, label = "", linewidth = 1, markersize = 2, color = color, alpha = 0.8)
    end
    return p1
end
# Plot the parameter traces for α and γ

param_trace (generic function with 1 method)

In [79]:
p_alpha = param_trace(res, 1; ground_truth_value = 0.1, xlabel = "Iteration", ylabel = "α", title = "Parameter Trace for α", run_idcs = best_idxs)
display(p_alpha)

In [80]:
p_beta = param_trace(res, 2; ground_truth_value = 0.02, xlabel = "Iteration", ylabel = "β", title = "Parameter Trace for β", run_idcs = best_idxs)

In [81]:
p_gamma = param_trace(res, 3; ground_truth_value = 0.3, xlabel = "Iteration", ylabel = "γ", title = "Parameter Trace for γ", run_idcs = best_idxs)

In [82]:
p_delta = param_trace(res, 4; ground_truth_value = 0.01, xlabel = "Iteration", ylabel = "δ", title = "Parameter Trace for δ", yscale = :log10, legend = :bottomright, run_idcs = best_idxs)

In [83]:
logmod(x) = sign(x) * log(abs(x) + 1)
p3 = plot(yscale = :log10, xlabel = "Iteration", ylabel = "nllh", title = "Error trace", yticks = 10.0 .^ (-2:9), ylim = (1e-2, 1e9))
scatter!(p3, best_res[1].ftrace, label = "nllh", markersize = 4, color = :blue, alpha = 0.8)

In [84]:
likelogmod(x) = sign(x) * log(abs(x) + 1)

# Choose some representative y values for ticks (in the original space)
ytick_vals = [-100,-10,-1,0, 1, 10, 100, 1e3, 1e4, 1e5, 1e6, 1e7, 1e8, 1e9]
ytick_labels = string.(ytick_vals)
ytick_positions = likelogmod.(ytick_vals)
colors = [:blue, :green, :orange, :purple, :cyan, :magenta, :yellow, :brown]
markers = [:v, :cross, :star, :triangle, :x, :diamond]
p3 = plot(xlabel = "Iteration", ylabel = "nllh", title = "Negative Log-Likelihood Trace",
          yticks = (ytick_positions, ytick_labels), ylim = (likelogmod(-100),likelogmod(1e9)))
for (i, cur_res) in enumerate(best_res)
    color = colors[i % length(colors) + 1]
    marker = markers[i % length(markers) + 1]
    scatter!(p3, likelogmod.(cur_res.ftrace), markersize = 4, label = "Run $i", color = color, marker = marker, alpha = 0.8)
    #plot the line connecting the points
    plot!(p3, likelogmod.(cur_res.ftrace), label = "", linewidth = 1, markersize = 2, color = color, alpha = 0.8)
end
display(p3)

In [85]:
run

run (generic function with 6 methods)

In [92]:
#plot the objective functions
p3 = plot()
for (i, cur_run) in enumerate(best_res[1:3])
    color = colors[i % length(colors) + 1]
    marker = markers[i % length(markers) + 1]
    ftrace = hcat(cur_run.ftrace ...)[1,:]
    scatter!(p3, ftrace, label = "Run $i", markersize = 4, color = color, marker = marker, alpha = 0.8)
end
plot!(p3, xlabel = "Iteration", ylabel = "Objective Function", title = "Objective Function Trace for Best Runs", xlim = (0,20))

In [87]:
# # using Plots; pythonplot()
# petab_prob.nllh(res.xmin)
# pval = copy(res.xmin)
# dist = 0.1
# alpha_range = range(res.xmin.α - dist, res.xmin.α + dist, length = 20)
# beta_range = range(res.xmin.β - dist, res.xmin.β + dist, length = 20)
# nllh_values = zeros(length(alpha_range), length(beta_range))
# for (i, α) in enumerate(alpha_range)
#     for (j, β) in enumerate(beta_range)
#         pval.α = α
#         pval.β = β
#         nllh_values[i, j] = petab_prob.nllh(pval)
#     end
# end     

In [88]:
# heatmap(alpha_range, beta_range, nllh_values, xlabel="α", ylabel="β", title="NLLH Heatmap", color=:viridis, c=:auto, xlims=(alpha_range[1], alpha_range[end]), ylims=(beta_range[1], beta_range[end]),
#         colorbar_title="NLLH", colorbar_ticks=:auto, colorbar_labelsize=10, colorbar_titlefontsize=12, scale=:log10)

## Neural ODEs

In [89]:
# test_case = "001"

# nn1 = @compact(
#     layer1 = Dense(2, 5, Lux.tanh),
#     layer2 = Dense(5, 5, Lux.tanh),
#     layer3 = Dense(5, 1)
# ) do x
#     embed = layer1(x)
#     embed = layer2(embed)
#     out = layer3(embed)
#     @return out
# end
# ml_models = Dict(:net1 => MLModel(nn1; static = false))
# # path_h5 = joinpath(@__DIR__, "test_cases", "hybrid", test_case, "petab", "net1_ps.hdf5")
# pnn = Lux.initialparameters(rng, nn1) |> ComponentArray |> f64
# # PEtab.set_ml_model_ps!(pnn, path_h5, nn1)

# function _lv1!(du, u, p, t, ml_models)
#     prey, predator = u
#     @unpack alpha, delta, beta = p
#     net1 = ml_models[:net1]
#     du_nn, st = net1.model([prey, predator], p[:net1], net1.st)
#     net1.st = st

#     du[1] = alpha*prey - beta * prey * predator # prey
#     du[2] = du_nn[1] - delta*predator # predator
#     return nothing
# end
# lv1! = let _ml_models = ml_models
#     (du, u, p, t) -> _lv1!(du, u, p, t, _ml_models)
# end

# p_mechanistic = (alpha = 0.1, beta = 0.02, delta = 0.01)
# p_ode = ComponentArray(merge(p_mechanistic, (net1=pnn,)))
# u0 = ComponentArray(prey = 40.0, predator = 9.0)
# uprob = ODEProblem(lv1!, u0, (0.0, 200.0), p_ode)

# p_alpha = PEtabParameter(:alpha; scale = :lin, lb = 0.0, ub = 1.0, value = 0.1)
# p_beta = PEtabParameter(:beta; scale = :lin, lb = 0.0, ub = 1.0, value = 0.02)
# p_delta = PEtabParameter(:delta; scale = :lin, lb = 0.0, ub = 1.0, value = 0.01)
# p_net1 = PEtabMLParameter(:net1, true, pnn)
# pest = [p_alpha, p_beta, p_delta, p_net1]

# obs_prey = PEtabObservable(:prey, 0.2)
# obs_predator = PEtabObservable(:predator, 0.2)
# obs = Dict("prey_o" => obs_prey, "predator_o" => obs_predator)

# conds = Dict("cond1" => Dict{Symbol, Symbol}())

# # path_m = joinpath(@__DIR__, "test_cases", "hybrid", test_case, "petab", "measurements.tsv")
# # measurements = CSV.read(path_m, DataFrame)

# model = PEtabModel(uprob, obs, measurements, pest; ml_models = ml_models,
#                    simulation_conditions = conds, verbose = true)

# osolver = ODESolver(Rodas5P(autodiff = false), abstol = 1e-10, reltol = 1e-10)
# petab_prob = PEtabODEProblem(model; odesolver = osolver, gradient_method = :ForwardDiff,
#                              split_over_conditions = true)

# ms_res = calibrate_multistart(petab_prob, IPNewton(), 2; seed = 1234)

### Datafitting

In [90]:
# ms_res = calibrate_multistart(petab_prob, IPNewton(), 50; nprocs = 2,
#                               dirsave="path_to_save_directory")

In [91]:
# using Optimization
# opt_prob = OptimizationProblem(petab_prob)
